In [1]:
!ls ../input/competitions/birdclef-2026

recording_location.txt	test_soundscapes  train_soundscapes
sample_submission.csv	train_audio	  train_soundscapes_labels.csv
taxonomy.csv		train.csv


In [2]:
BASE = "../input/competitions/birdclef-2026/"

In [ ]:
import os
import ast
import time
import glob
import torch
import torchaudio
import pandas as pd
import numpy as np
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
TRAIN_CSV = os.path.join(BASE, "train.csv")
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")

class Config:
    SR = 32000               
    DURATION = 5             
    MAX_LENGTH = SR * DURATION 
    N_MELS = 128             
    N_FFT = 1024
    HOP_LENGTH = 512
    BATCH_SIZE = 32
    NUM_WORKERS = 4
    EPOCHS = 10

train_df = pd.read_csv(TRAIN_CSV)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)

CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_df['target'] = train_df['primary_label'].map(class_to_idx)

In [ ]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, config, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=config.SR,
            n_fft=config.N_FFT,
            hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS,
            f_min=50,
            f_max=14000
        )
        
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        waveform, sr = torchaudio.load(audio_path)
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        
        if audio_len > self.config.MAX_LENGTH:
            if self.is_train:
                max_start = audio_len - self.config.MAX_LENGTH
                start = np.random.randint(0, max_start)
            else:
                start = (audio_len - self.config.MAX_LENGTH) // 2
            waveform = waveform[:, start:start + self.config.MAX_LENGTH]
            
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        mel_spec = self.mel_transform(waveform)
        mel_spec = self.amp_to_db(mel_spec)
        
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1
        
        mel_spec = mel_spec.expand(3, -1, -1)
        
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        
        primary_idx = class_to_idx.get(row['primary_label'])
        if primary_idx is not None:
            target[primary_idx] = 1.0
            
        secondary_labels = ast.literal_eval(row.get('secondary_labels', "[]"))
        for sec_label in secondary_labels:
            sec_idx = class_to_idx.get(sec_label)
            if sec_idx is not None:
                target[sec_idx] = 1.0

        return mel_spec, target

In [ ]:
train_split, val_split = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42 
)

train_dataset = BirdCLEFDataset(train_split, TRAIN_AUDIO_DIR, Config, is_train=True)
val_dataset = BirdCLEFDataset(val_split, TRAIN_AUDIO_DIR, Config, is_train=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

In [ ]:
class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=True):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0 
        )
        
        in_features = self.backbone.num_features
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFModel(num_classes=NUM_CLASSES).to(device)
print(f"Model loaded on {device}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model loaded on cuda


In [ ]:
def calculate_competition_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) == 2:
            class_auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(class_auc)
    if len(aucs) == 0:
        return 0.5
    return np.mean(aucs)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)

In [ ]:
best_val_auc = 0.0

for epoch in range(Config.EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]", leave=False)
    
    for images, targets in train_pbar:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        logits = model(images)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        train_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    all_val_targets = []
    all_val_preds = []
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, targets in val_pbar:
            images = images.to(device)
            targets = targets.to(device)
            
            logits = model(images)
            loss = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits)
            
            all_val_targets.append(targets.cpu().numpy())
            all_val_preds.append(probs.cpu().numpy())
            
            val_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
    val_loss = val_loss / len(val_loader.dataset)
    
    all_val_targets = np.vstack(all_val_targets)
    all_val_preds = np.vstack(all_val_preds)
    
    val_auc = calculate_competition_roc_auc(all_val_targets, all_val_preds)
    
    # Logging
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{Config.EPOCHS} | LR: {current_lr:.2e}")
    print(f"  -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")
    
    if val_auc > best_val_auc:
        print(f"  [+] Validation AUC improved ({best_val_auc:.4f} -> {val_auc:.4f}). Saving model!")
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_birdclef_model.pth')
        
print("Training Complete!")

In [ ]:
model.load_state_dict(torch.load('best_birdclef_model.pth'))
model.to(device)
model.eval()